# Pokémon Type Classifier
Multi-label classifier using official artwork from `pokemon_data.csv`.

## Imports & Setup

In [1]:
import os
import csv
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import requests
from io import BytesIO

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

Using device: cuda


## 2. Pokémon Types

In [2]:
POKEMON_TYPES = [
    "normal", "fire", "water", "grass", "electric", "ice",
    "fighting", "poison", "ground", "flying", "psychic",
    "bug", "rock", "ghost", "dark", "dragon", "steel", "fairy"
]

## 3. Dataset Class

In [3]:
class PokemonDataset(Dataset):
    def __init__(self, csv_file, transform=None, img_dir='pokemon_images'):
        self.rows = list(csv.DictReader(open(csv_file)))
        self.transform = transform
        self.img_dir = img_dir
        os.makedirs(img_dir, exist_ok=True)

    def load_sprite(self, url, local_name):
        if not url:
            return Image.new('RGBA', (224, 224), (0, 0, 0, 0))

        filepath = os.path.join(self.img_dir, local_name)
        if not os.path.exists(filepath):
            try:
                img_data = requests.get(url).content
                with open(filepath, 'wb') as f:
                    f.write(img_data)
            except:
                return Image.new('RGBA', (224, 224), (0, 0, 0, 0))

        return Image.open(filepath).convert('RGB')

    def encode_types(self, type_string):
        labels = type_string.split(", ")
        return torch.tensor([1 if t in labels else 0 for t in POKEMON_TYPES], dtype=torch.float32)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        url = row['official_artwork']
        image = self.load_sprite(url, f"{row['id']}.png")
        if self.transform:
            image = self.transform(image)
        return image, self.encode_types(row['types'])

## 4. Data Transforms & DataLoader

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset = PokemonDataset('pokemon_data.csv', transform)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

print('Dataset size:', len(dataset))

Dataset size: 1025


## 5. Model Definition

In [5]:
def build_model():
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(512, len(POKEMON_TYPES))
    return model

model = build_model().to(DEVICE)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/michelle/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100.0%


## 6. Training Loop

In [6]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 5  # adjust as needed

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0

    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {running_loss:.4f}")

torch.save(model.state_dict(), 'pokemon_model.pt')
print('Model saved as pokemon_model.pt')

Epoch 1/5 - Loss: 24.1407
Epoch 2/5 - Loss: 13.0028
Epoch 3/5 - Loss: 8.8427
Epoch 4/5 - Loss: 5.5375
Epoch 5/5 - Loss: 3.7914
Model saved as pokemon_model.pt


## 7. Prediction / Inference

In [8]:
def predict(model, image_path, threshold=0.5):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])
    img = Image.open(image_path).convert('RGB')
    img = transform(img).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(img)
        probs = torch.sigmoid(logits)[0]

    results = [(t, float(p)) for t, p in zip(POKEMON_TYPES, probs) if p > threshold]
    return results

# Example usage
model.load_state_dict(torch.load('pokemon_model.pt', map_location=DEVICE))
image_path = 'test_images/char-pikachu.png'  # Pikachu example
predicted = predict(model, image_path)
print('Predicted types:', predicted)

Predicted types: []


/home/michelle/Documents/SchoolFiles/AI/pokemonProject/venv/lib/python3.12/site-packages/PIL/Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
